# FinReasoningAI Colab
End-to-end FinCoT QLoRA training for `Qwen/Qwen2.5-14B-Instruct`, with a standalone PEFT adapter saved for vLLM LoRA serving.

## Step 0a: GPU Check
Expected runtime: under 1 minute. VRAM guidance: training is designed for an A100 with at least 35 GB free VRAM before model load.

In [ ]:
import subprocess

def gpu_summary():
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.free",
            "--format=csv,noheader,nounits",
        ], text=True)
        print(out.strip())
        first = out.strip().splitlines()[0].split(",")
        free_gb = float(first[2].strip()) / 1024.0
        if free_gb < 35:
            print(f"Warning: only {free_gb:.1f} GB free VRAM detected; training may OOM.")
    except Exception as exc:
        print(f"Unable to query GPU details: {exc}")

gpu_summary()

NVIDIA A100-SXM4-80GB, 81920, 81153


## Step 0b: Mount Google Drive and Infer Workspace
Expected runtime: 1-2 minutes. This cell mounts Drive and auto-detects the notebook workspace instead of hardcoding paths.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/FinReasoningAI"
PROJECT_DIR = f"{DRIVE_BASE}/FinReasoningAI"

print("DRIVE_BASE:", DRIVE_BASE)
print("PROJECT_DIR:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_BASE: /content/drive/MyDrive/FinReasoningAI
PROJECT_DIR: /content/drive/MyDrive/FinReasoningAI/FinReasoningAI


## Step 0c: Install Missing Dependencies
Expected runtime: 3-8 minutes on a fresh runtime. PyTorch is intentionally not installed here because Colab already provides the GPU build.

In [ ]:
import importlib
import pkg_resources
import subprocess
import sys

REQUIRED = {
    "transformers":  "4.41.0",
    "datasets":      "2.19.0",
    "accelerate":    "0.30.0",
    "peft":          "0.10.0",
    "trl":           "0.8.6",
    "bitsandbytes":  "0.44.0",
    "evaluate":      "0.4.1",
    "rouge-score":   "0.1.2",
    "scikit-learn":  "1.4.0",
    "pandas":        "2.2.0",
    "pydantic":      "2.7.0",
    "jsonlines":     "4.0.0",
    "bitsandbytes": "0.46.1"
}

missing = []
for package, version in REQUIRED.items():
    try:
        installed = pkg_resources.get_distribution(package).version
        if installed != version:
            missing.append(f"{package}=={version}")
    except Exception:
        missing.append(f"{package}=={version}")

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Restart the runtime before training if packages were freshly installed.")
else:
    print("All required packages already match the requested versions.")

All required packages already match the requested versions.


/tmp/ipykernel_3610/3137331769.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


## Step 0d: Clone or Pull the Repository
Expected runtime: under 2 minutes. This cell always refreshes the repo in the current Colab workspace before imports.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/juankim834/FinReasoningAI.git"
DRIVE_BASE = "/content/drive/MyDrive/FinReasoningAI"
PROJECT_DIR = Path(DRIVE_BASE) / "FinReasoningAI"

if (PROJECT_DIR / ".git").exists():
    print("⏳ 正在执行 git pull --ff-only...")

    # 1. 使用 subprocess.run 显式捕获输出
    pull_result = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        capture_output=True,
        text=True
    )

    # 2. 打印 Git 的真实输出日志
    if pull_result.stdout:
        print("[Git 输出]:\n" + pull_result.stdout.strip())
    if pull_result.stderr:
        print("[Git 信息/报错]:\n" + pull_result.stderr.strip())

    # 检查是否报错
    if pull_result.returncode != 0:
        raise RuntimeError(f"❌ Git Pull 失败！返回码: {pull_result.returncode}")

    # 3. 打印当前最新的一条 commit，双重确认代码状态
    log_result = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "log", "-1", "--oneline"],
        capture_output=True,
        text=True
    )
    print(f"✅ 当前代码版本: {log_result.stdout.strip()}")

else:
    raise FileNotFoundError(
        f"Expected repo at {PROJECT_DIR}, but .git was not found. "
        "Please ensure the real repository lives in DRIVE_BASE/FinReasoningAI."
    )

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("📂 PROJECT_DIR 加载完毕:", PROJECT_DIR)

⏳ 正在执行 git pull --ff-only...
[Git 输出]:
Updating 9f6ab40..f3a0398
Fast-forward
 FinReasoningAI_Eval.ipynb | 428 +++++++++++++++++++++++++++++++++-------------
 src/train/sft_train.py    |   4 +
 2 files changed, 311 insertions(+), 121 deletions(-)
[Git 信息/报错]:
From https://github.com/juankim834/FinReasoningAI
   9f6ab40..f3a0398  main       -> origin/main
✅ 当前代码版本: f3a0398 fix no saving on step 100
📂 PROJECT_DIR 加载完毕: /content/drive/MyDrive/FinReasoningAI/FinReasoningAI


## Step 1: Configuration
Expected runtime: immediate. Adjust the knobs here before running the data, training, and evaluation cells below.

In [ ]:
# MODEL
MODEL_ID          = "Qwen/Qwen2.5-14B-Instruct"

# DATA
MAX_SAMPLES       = None            # None = full SFT split
INCLUDE_COT       = True
FINAL_SAMPLE_SIZE = 1800
TRAIN_SIZE        = 1500
TEST_SIZE         = 300
SEED              = 42
PROCESSED_DATA_DIR = "data/processed_fincot_sft"

# TRAINING
NUM_EPOCHS        = 1
BATCH_SIZE        = 1
GRAD_ACCUM        = 16
LEARNING_RATE     = 1e-4
MAX_SEQ_LEN       = 4096
USE_WANDB         = False
GRAD_CKPT         = True
MAX_TRAIN_LEN = 4096

# LoRA
LORA_R            = 64
LORA_ALPHA        = 128
LORA_DROPOUT      = 0.05
LORA_TARGET_MODS = [
    "q_proj", "v_proj",
]

# OUTPUT
OUTPUT_DIR        = "outputs/sft_qlora_second"
ADAPTER_SAVE_DIR  = f"{OUTPUT_DIR}/final_adapter"
EVAL_MAX_SAMPLES  = 200
SKIP_DPO          = True
os.environ["FINREASONING_USE_BF16"] = "1"



## Step 2a: Load the Base Model
Expected runtime: 5-10 minutes. VRAM usage after 4-bit load is typically around 10-15 GB before LoRA adapters and activations.

In [ ]:
from src.model.load_model import DEFAULT_BNB_CONFIG, load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    model_id=MODEL_ID,
    bnb_config=DEFAULT_BNB_CONFIG,
)

import gc
import torch
if torch.cuda.is_available():
    print(f"Allocated VRAM after base load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Base model loader check passed. Training and eval cells will load models explicitly when needed.")



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Allocated VRAM after base load: 10.84 GB
Base model loader check passed. Training and eval cells will load models explicitly when needed.


## Step 2b: Apply QLoRA
Expected runtime: under 2 minutes. This attaches trainable adapters while keeping the base model quantized for A100-friendly fine-tuning.

In [ ]:
from src.model.apply_lora import apply_qlora
from src.model.load_model import DEFAULT_BNB_CONFIG, load_model_and_tokenizer
from src.train.sft_train import build_lora_config

model, tokenizer = load_model_and_tokenizer(
    model_id=MODEL_ID,
    bnb_config=DEFAULT_BNB_CONFIG,
)

lora_config = build_lora_config(
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODS,
)
model = apply_qlora(model, lora_config=lora_config, gradient_checkpointing=True)
model.print_trainable_parameters()


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

trainable params: 50,331,648 || all params: 14,820,365,312 || trainable%: 0.3396113856872788


## Step 3a: Load FinCoT Samples
Expected runtime: 1-5 minutes depending on whether the dataset is pulled from Hugging Face or read locally.

In [ ]:
from src.data.fincot_loader import load_fincot_samples

samples, dataset_meta = load_fincot_samples(max_samples=MAX_SAMPLES)
print(dataset_meta)
assert len(samples) > 0
assert all(key in samples[0] for key in ["question", "answer", "reasoning"])



{'dataset_name': 'TheFinAI/FinCoT', 'split_name': 'SFT', 'columns': ['Question', 'Reasoning_process', 'Final_response', 'Negative_reasoning_process', 'Negative_response'], 'original_sample_count': 7686}


## Step 3b: Inspect One Sample per Task
Expected runtime: under 1 minute. This helps confirm the loader is producing the normalized schema we expect before formatting.

In [ ]:
print(samples[0])



{'Question': 'Please answer the given financial question based on the context.\nContext: amortization expense , which is included in selling , general and administrative expenses , was $ 13.0 million , $ 13.9 million and $ 8.5 million for the years ended december 31 , 2016 , 2015 and 2014 , respectively . the following is the estimated amortization expense for the company 2019s intangible assets as of december 31 , 2016 : ( in thousands ) .\n|2017|$ 10509|\n|2018|9346|\n|2019|9240|\n|2020|7201|\n|2021|5318|\n|2022 and thereafter|16756|\n|amortization expense of intangible assets|$ 58370|\nat december 31 , 2016 , 2015 and 2014 , the company determined that its goodwill and indefinite- lived intangible assets were not impaired . 5 . credit facility and other long term debt credit facility the company is party to a credit agreement that provides revolving commitments for up to $ 1.25 billion of borrowings , as well as term loan commitments , in each case maturing in january 2021 . as of d

## Step 3c: Build Prompt/Completion Datasets
Expected runtime: 2-5 minutes. The preprocessing path uses `tokenizer.apply_chat_template` so the chat format stays aligned with Qwen tokenizer updates.

In [ ]:
import numpy, bitsandbytes, torch
print("numpy:", numpy.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("cuda:", torch.cuda.is_available())

numpy: 1.26.4
bitsandbytes: 0.46.1
cuda: True


In [ ]:
from pathlib import Path

from src.data.preprocess import prepare_fincot_sft_dataset
from src.train.sft_train import load_datasets, _tokenize_old_trl_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

if not Path(PROCESSED_DATA_DIR).exists():
    print(f"{PROCESSED_DATA_DIR} not found. Building the processed dataset first...")
    prepare_fincot_sft_dataset(
        tok,
        output_dir=PROCESSED_DATA_DIR,
        final_sample_size=FINAL_SAMPLE_SIZE,
        train_size=TRAIN_SIZE,
        test_size=TEST_SIZE,
        include_cot=INCLUDE_COT,
        seed=SEED,
    )

ds = load_datasets(PROCESSED_DATA_DIR)
tmp = _tokenize_old_trl_dataset(ds["train"], tok, 4096)
print(tmp.column_names)
print({k: type(v).__name__ for k, v in tmp[0].items()})





Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


['input_ids', 'attention_mask', 'special_tokens_mask']
{'input_ids': 'list', 'attention_mask': 'list', 'special_tokens_mask': 'list'}


In [ ]:
from src.data.preprocess import prepare_fincot_sft_dataset, print_preparation_summary, format_sample_as_chat

dataset, dataset_summary = prepare_fincot_sft_dataset(
    tokenizer,
    output_dir=PROCESSED_DATA_DIR,
    final_sample_size=FINAL_SAMPLE_SIZE,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    include_cot=INCLUDE_COT,
    seed=SEED,
)
print_preparation_summary(dataset_summary, PROCESSED_DATA_DIR)
print({split: len(ds) for split, ds in dataset.items()})
example = format_sample_as_chat(samples[0], tokenizer, include_cot=INCLUDE_COT)
print(example["prompt"][:1500])
print("\n--- COMPLETION ---\n")
print(example["completion"])
assert example["prompt"]
assert example["completion"]



Saving the dataset (0/1 shards):   0%|          | 0/1500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset: TheFinAI/FinCoT
Loaded split: SFT
Original number of samples in SFT Training: 7686
Available dataset columns: ['Question', 'Reasoning_process', 'Final_response', 'Negative_reasoning_process', 'Negative_response']
Number of Numerical Reasoning samples: 7603
Number of Non-Numerical Reasoning samples: 83
Final training set size: 1500
Final test set size: 300
Category distribution in train: {'Numerical Reasoning': 1431, 'Non-Numerical Reasoning': 69}
Category distribution in test: {'Numerical Reasoning': 286, 'Non-Numerical Reasoning': 14}
Saved processed dataset to: data/processed_fincot_sft
{'train': 1500, 'test': 300}
Please answer the given financial question based on the context.
Context: amortization expense , which is included in selling , general and administrative expenses , was $ 13.0 million , $ 13.9 million and $ 8.5 million for the years ended december 31 , 2016 , 2015 and 2014 , respectively . the following is the estimated amortization expense for the company 2019s 

## Step 4: SFT Training (3 Epoch)
Expected runtime: several hours on an A100 depending on dataset size. Peak VRAM will usually sit in the 30-40 GB range with the default sequence length and effective batch size.

In [ ]:
from src.train.sft_train import load_datasets, _prepare_old_trl_dataset
ds = load_datasets(PROCESSED_DATA_DIR)
tmp = _prepare_old_trl_dataset(ds["train"], tokenizer=tok)
print(tmp.column_names)
print(tmp[0].keys())



['text']
dict_keys(['text'])


In [ ]:
import importlib
import src.train.sft_train as sft_train
importlib.reload(sft_train)

from src.train.sft_train import main as sft_main, save_adapter_for_vllm

trainer = sft_main(
    model_id=MODEL_ID,
    output_dir=OUTPUT_DIR,
    data_dir=PROCESSED_DATA_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_seq_length=MAX_SEQ_LEN,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODS,
    use_wandb=USE_WANDB,
    gradient_checkpointing=GRAD_CKPT,
    max_train_length=MAX_TRAIN_LEN,
    eval_steps=22,
    save_steps=22,
)
save_adapter_for_vllm(trainer=trainer, output_dir=ADAPTER_SAVE_DIR)



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
22,1.175000,1.069613
44,0.981200,0.921543
66,0.906200,0.889430
88,0.881200,0.884071


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in ver

Adapter saved to: outputs/sft_qlora_second/final_adapter

To serve with vLLM:
  vllm serve Qwen/Qwen2.5-14B-Instruct \
    --enable-lora \
    --lora-modules fin-reasoning=outputs/sft_qlora_second/final_adapter \
    --max-lora-rank 64

To call with LoRARequest:
  from vllm import LLM, SamplingParams
  from vllm.lora.request import LoRARequest
  llm = LLM("Qwen/Qwen2.5-14B-Instruct", enable_lora=True)
  outputs = llm.generate(prompts, SamplingParams(...),
                         lora_request=LoRARequest("fin-reasoning", 1, "outputs/sft_qlora_second/final_adapter"))


In [ ]:
from google.colab import runtime
runtime.unassign()

## Step 5: Evaluation
Expected runtime: 10-30 minutes for 200 samples depending on decoding mode. This reuses the raw held-out test slice so answer/context fields are available for metrics.

In [ ]:
from pathlib import Path

import gc
import torch

from src.data.preprocess import load_eval_test_samples
from src.eval.evaluate import evaluate_model
from src.model.load_model import load_model_with_adapter

if "trainer" in globals():
    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

eval_model, eval_tokenizer = load_model_with_adapter(MODEL_ID, ADAPTER_SAVE_DIR)
test_samples = load_eval_test_samples(PROCESSED_DATA_DIR)
metrics = evaluate_model(
    eval_model,
    eval_tokenizer,
    test_samples,
    output_csv="outputs/eval_results.csv",
    max_samples=EVAL_MAX_SAMPLES,
)
print(metrics)



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.


KeyboardInterrupt: 

## Step 6a: Direct Inference Demo
Expected runtime: under 1 minute for a single prompt. This is the shortest path for a basic financial QA response.

In [ ]:
from src.inference.generate import build_prompt, generate_answer
from src.model.load_model import load_model_with_adapter

question = "What does a lower debt-to-equity ratio generally suggest about a company's balance sheet?"
if "eval_model" not in globals() or "eval_tokenizer" not in globals():
    eval_model, eval_tokenizer = load_model_with_adapter(MODEL_ID, ADAPTER_SAVE_DIR)
prompt = build_prompt(question=question, tokenizer=eval_tokenizer)
print(prompt[:1000])
print()
print(generate_answer(eval_model, eval_tokenizer, question=question, max_new_tokens=128, grounding_check=False))



## Step 6b: Chain-of-Thought Inference Demo
Expected runtime: under 1 minute. This uses the reasoning-oriented inference path and expects the answer to end with an `Answer:` line.

In [ ]:
cot_question = "If revenue grows from 120 to 150, what is the percentage growth?"
print(generate_answer(model, tokenizer, question=cot_question, use_cot=True, max_new_tokens=256, grounding_check=False))

## Step 6c: Tool-Augmented Inference Demo
Expected runtime: under 1 minute. This exercises the tool loop so we can confirm a ratio question either triggers a tool call or still returns a plain answer without crashing.

In [ ]:
from src.inference.generate import generate_with_tools
from src.model.load_model import load_model_with_adapter
from tools.financial_tools import FINANCIAL_TOOLS

if "eval_model" not in globals() or "eval_tokenizer" not in globals():
    eval_model, eval_tokenizer = load_model_with_adapter(MODEL_ID, ADAPTER_SAVE_DIR)

tool_demo = generate_with_tools(
    eval_model,
    eval_tokenizer,
    question="What is Apple's P/E ratio given net income of 97 and market cap of 2800?",
    tools=FINANCIAL_TOOLS,
    max_new_tokens=256,
)
print(tool_demo)



## Step 6d: Self-Consistency Demo
Expected runtime: a few minutes for 5 samples because each prompt is sampled multiple times. Use this to inspect agreement-based robustness after training.

In [ ]:
from src.inference.self_consistency import sample_with_self_consistency

prompt = build_prompt(question="What is the CAGR from 100 to 121 over 2 periods?", tokenizer=tokenizer, use_cot=True)
final_answer, confidence, raw_answers = sample_with_self_consistency(
    model, tokenizer, prompt, n=5, temperature=0.7, max_new_tokens=128
)
print("Final:", final_answer)
print("Confidence:", confidence)
print(raw_answers)

## Step 7: Optional DPO
Expected runtime: only relevant if you later choose to extend the pipeline beyond the 1-epoch SFT run. This stays disabled by default.

In [ ]:
if SKIP_DPO:
    print("Skipping DPO as configured.")
else:
    print("Add your optional DPO workflow here.")

## Verification Checklist
Expected runtime: under 1 minute after earlier cells complete. These assertions cover the minimum smoke tests requested in the rewrite spec.

In [ ]:
from pathlib import Path

assert len(samples) > 0
formatted = format_sample_as_chat(samples[0], tokenizer, include_cot=INCLUDE_COT)
assert formatted["prompt"] and formatted["completion"]
assert set(dataset.keys()) == {"train", "test"}
assert sum(len(ds) for ds in dataset.values()) == FINAL_SAMPLE_SIZE
assert Path(ADAPTER_SAVE_DIR).exists()
assert Path(ADAPTER_SAVE_DIR, "adapter_config.json").exists()
assert isinstance(tool_demo["tool_calls"], list)
print("Notebook smoke checks passed.")



In [ ]:
from google.colab import runtime
runtime.unassign()